Imports and paths

In [8]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

raw_path = project_root / "data" / "raw" / "existing_dataset" / "data.csv"

processed_dir = project_root / "data" / "processed" / "existing_dataset"
processed_dir.mkdir(parents=True, exist_ok=True)

processed_path = processed_dir / "completed_students.csv"

print("Project root:", project_root)
print("Dataset exists:", raw_path.exists())

Project root: /Users/awais/Desktop/COMP8240_Project
Dataset exists: True


Load the dataset

In [9]:
df = pd.read_csv(raw_path, sep=";")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (4424, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


Inspect columns 

In [10]:
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

 1. Marital status
 2. Application mode
 3. Application order
 4. Course
 5. Daytime/evening attendance	
 6. Previous qualification
 7. Previous qualification (grade)
 8. Nacionality
 9. Mother's qualification
10. Father's qualification
11. Mother's occupation
12. Father's occupation
13. Admission grade
14. Displaced
15. Educational special needs
16. Debtor
17. Tuition fees up to date
18. Gender
19. Scholarship holder
20. Age at enrollment
21. International
22. Curricular units 1st sem (credited)
23. Curricular units 1st sem (enrolled)
24. Curricular units 1st sem (evaluations)
25. Curricular units 1st sem (approved)
26. Curricular units 1st sem (grade)
27. Curricular units 1st sem (without evaluations)
28. Curricular units 2nd sem (credited)
29. Curricular units 2nd sem (enrolled)
30. Curricular units 2nd sem (evaluations)
31. Curricular units 2nd sem (approved)
32. Curricular units 2nd sem (grade)
33. Curricular units 2nd sem (without evaluations)
34. Unemployment rate
35. Inflation 

Inspect target distribution

In [11]:
print(df["Target"].value_counts())

Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


Check missing values and duplicates

In [12]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values: 0
Duplicate rows: 0


Validate the original dataset

In [13]:
assert df.shape == (4424, 37)

expected_targets = {"Graduate", "Dropout", "Enrolled"}

assert set(df["Target"].unique()) == expected_targets

print("Original dataset validation: PASSED")

Original dataset validation: PASSED


Remove currently enrolled students

In [14]:
df_completed = df[
    df["Target"].isin(["Graduate", "Dropout"])
].copy()

print("Completed dataset shape:", df_completed.shape)

print("\nTarget distribution:")
print(df_completed["Target"].value_counts())

Completed dataset shape: (3630, 37)

Target distribution:
Target
Graduate    2209
Dropout     1421
Name: count, dtype: int64


Create our binary outcome

In [15]:
df_completed["Outcome"] = df_completed["Target"].map({
    "Graduate": "Success",
    "Dropout": "Failure"
})

print(df_completed["Outcome"].value_counts())

Outcome
Success    2209
Failure    1421
Name: count, dtype: int64


Validate the binary outcome

In [16]:
assert len(df_completed) == 3630
assert df_completed["Outcome"].isnull().sum() == 0

assert df_completed["Outcome"].value_counts()["Success"] == 2209
assert df_completed["Outcome"].value_counts()["Failure"] == 1421

print("Binary outcome validation: PASSED")

Binary outcome validation: PASSED


Remove original Target

In [17]:
df_model = df_completed.drop(columns=["Target"]).copy()

print("Prepared dataset shape:", df_model.shape)
df_model.head()

Prepared dataset shape: (3630, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Outcome
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Failure
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Success
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Failure
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Success
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Success


clean column names

In [25]:
df_prepared = df_model.copy()

# Remove accidental leading/trailing whitespace from column names
df_prepared.columns = df_prepared.columns.str.strip()

print("Dataset shape:", df_prepared.shape)

Dataset shape: (3630, 37)


define categorical predictors

In [26]:
categorical_cols = [
    "Marital status",
    "Application mode",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International"
]

print("Number of categorical predictors:", len(categorical_cols))

Number of categorical predictors: 17


Inspect data types

In [27]:
for col in categorical_cols:
    print(f"{col:35} {df_prepared[col].nunique():3} unique values")

Marital status                        6 unique values
Application mode                     18 unique values
Course                               17 unique values
Daytime/evening attendance            2 unique values
Previous qualification               17 unique values
Nacionality                          19 unique values
Mother's qualification               29 unique values
Father's qualification               34 unique values
Mother's occupation                  29 unique values
Father's occupation                  42 unique values
Displaced                             2 unique values
Educational special needs             2 unique values
Debtor                                2 unique values
Tuition fees up to date               2 unique values
Gender                                2 unique values
Scholarship holder                    2 unique values
International                         2 unique values


create predictors and binary target

In [28]:
X_raw = df_prepared.drop(columns=["Outcome"])

Y = df_prepared["Outcome"].map({
    "Failure": 0,
    "Success": 1
})

print("X raw shape:", X_raw.shape)
print("Y shape:", Y.shape)

print("\nTarget distribution:")
print(Y.value_counts())

X raw shape: (3630, 36)
Y shape: (3630,)

Target distribution:
Outcome
1    2209
0    1421
Name: count, dtype: int64


one-hot encode categorical predictors

In [29]:
X_encoded = pd.get_dummies(
    X_raw,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print("Before encoding:", X_raw.shape)
print("After encoding :", X_encoded.shape)

Before encoding: (3630, 36)
After encoding : (3630, 229)


inspect encoded predictors

In [30]:
print("First 25 predictor names:")

for col in X_encoded.columns[:25]:
    print(col)

First 25 predictor names:
Application order
Previous qualification (grade)
Admission grade
Age at enrollment
Curricular units 1st sem (credited)
Curricular units 1st sem (enrolled)
Curricular units 1st sem (evaluations)
Curricular units 1st sem (approved)
Curricular units 1st sem (grade)
Curricular units 1st sem (without evaluations)
Curricular units 2nd sem (credited)
Curricular units 2nd sem (enrolled)
Curricular units 2nd sem (evaluations)
Curricular units 2nd sem (approved)
Curricular units 2nd sem (grade)
Curricular units 2nd sem (without evaluations)
Unemployment rate
Inflation rate
GDP
Marital status_2
Marital status_3
Marital status_4
Marital status_5
Marital status_6
Application mode_2


validation

In [31]:
print("Remaining object columns:")
print(X_encoded.select_dtypes(include=["object"]).columns.tolist())

print("\nMissing values:", X_encoded.isnull().sum().sum())
print("Rows:", X_encoded.shape[0])
print("Encoded predictors:", X_encoded.shape[1])

Remaining object columns:
[]

Missing values: 0
Rows: 3630
Encoded predictors: 229


standardise features

In [32]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_encoded)

print("Encoded shape:", X_encoded.shape)
print("Scaled shape :", X_scaled.shape)

Encoded shape: (3630, 229)
Scaled shape : (3630, 229)


validation

In [33]:
assert X_encoded.shape[0] == 3630
assert X_scaled.shape == X_encoded.shape
assert len(Y) == 3630

assert X_encoded.isnull().sum().sum() == 0
assert Y.isnull().sum() == 0

assert (Y == 1).sum() == 2209
assert (Y == 0).sum() == 1421

print("Feature preprocessing validation: PASSED")

print("\nFinal modelling data")
print("Students           :", X_scaled.shape[0])
print("Encoded predictors :", X_scaled.shape[1])
print("Success            :", (Y == 1).sum())
print("Failure            :", (Y == 0).sum())

Feature preprocessing validation: PASSED

Final modelling data
Students           : 3630
Encoded predictors : 229
Success            : 2209
Failure            : 1421


In [34]:
# Save the cleaned completed-student dataset

df_prepared.to_csv(processed_path, index=False)

print("Saved processed dataset to:")
print(processed_path)

saved_df = pd.read_csv(processed_path)

print("Saved dataset shape:", saved_df.shape)

assert saved_df.shape == (3630, 37)

print("Processed dataset save validation: PASSED")

Saved processed dataset to:
/Users/awais/Desktop/COMP8240_Project/data/processed/existing_dataset/completed_students.csv
Saved dataset shape: (3630, 37)
Processed dataset save validation: PASSED
